# All-23 RGB-Geodesic Cascade
Hucreleri sirasiyla calistirin. Once smoke test, ardindan A100 veya 5-fold CV kosusu yapin.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import subprocess

CODE_ROOT = Path('/content/comparative-study')
REPO_URL = 'https://github.com/eckdev/comparative-study.git'
if CODE_ROOT.exists():
    subprocess.run(['git', '-C', str(CODE_ROOT), 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(CODE_ROOT)], check=True)
subprocess.run(['pip', 'install', '-q', '-r', str(CODE_ROOT / 'all23_rgb_geodesic_cascade/requirements.txt')], check=True)
print('CODE_ROOT:', CODE_ROOT)

In [ ]:
paths = {
    'dataset': Path('/content/drive/MyDrive/orthodontic/data/dataset'),
    'split': CODE_ROOT / 'shared_splits/orthodontic_180_60_60_seed42.json',
    'legacy_transform': Path('/content/drive/MyDrive/orthodontic/transforms/orthodontic_procrustes_rigid_20260627_143801'),
    'agh_v6': Path('/content/drive/MyDrive/orthodontic/diffusion_runs/aghformer_v6_stage2_raw_fine_refiner_p12000'),
    'stacker': Path('/content/drive/MyDrive/orthodontic/diffusion_runs/shape_prior_stacker'),
}
for name, path in paths.items():
    print(f'{name:18s}', path.exists(), path)
assert paths['dataset'].exists(), 'Drive dataset bulunamadi'

## 1. Smoke test

In [ ]:
%cd /content/comparative-study/all23_rgb_geodesic_cascade
!python -u colab_run_all23_rgb_geodesic.py --preset smoke

## 2A. A100 sabit split ana kosu

In [ ]:
%cd /content/comparative-study/all23_rgb_geodesic_cascade
!python -u colab_run_all23_rgb_geodesic.py --preset a100 --seed 42

## 2B. Leakage-free 5-fold ROI preflight
Bu hucre egitim baslatmadan tum foldlarin ROI kapsamlarini denetler.

In [ ]:
%cd /content/comparative-study/all23_rgb_geodesic_cascade
!python -u colab_run_all23_rgb_geodesic.py --preset cv_preflight --seed 42

## 2C. Leakage-free 5-fold yayin kosusu
Preflight bes fold icin basarili olduktan sonra calistirin.

In [ ]:
%cd /content/comparative-study/all23_rgb_geodesic_cascade
!python -u colab_run_all23_rgb_geodesic.py --preset cv --seed 42

In [ ]:
import json
result = Path('/content/drive/MyDrive/orthodontic/all23_rgb_geodesic_runs/full_fixed_seed42/metrics_test.json')
if result.exists():
    metrics = json.loads(result.read_text())
    print('All-23 ALE:', metrics['overall']['ale'])
    print('Core20 ALE:', metrics['core20']['ale'])
    print('Hard3 ALE:', metrics['hard3']['ale'])
    print('Median:', metrics['overall']['median'])
else:
    print('Henuz sonuc dosyasi yok:', result)